# Benchmark Analysis

This notebook analyzes the real benchmark results produced by the AI Interview Coach across v3, v4, v5, and v6.

It complements the Python scripts in `src/scripts/` by presenting the results in an interactive, report-friendly format.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RESULTS_PATH = PROJECT_ROOT / "data" / "results" / "version_benchmark_results.csv"

benchmark_df = pd.read_csv(RESULTS_PATH)
benchmark_df.head()

## Map Version Labels

In [ ]:
VERSION_LABELS = {
    "v3_hybrid_semantic_flexible_keyword": "v3",
    "v4_hybrid_keyword_guardrails": "v4",
    "v5_llm_rubric": "v5",
    "v6_llm_structured_rubric": "v6",
}

benchmark_df["version_label"] = benchmark_df["scoring_version"].map(VERSION_LABELS)
benchmark_df[["scoring_version", "version_label"]].drop_duplicates().sort_values("version_label")

## Average Scores by Version

In [ ]:
benchmark_df.groupby("version_label")[["final_score", "rating"]].mean().round(3).sort_index()

In [ ]:
benchmark_df.groupby("version_label")[["final_score", "rating"]].mean().sort_index().plot(kind="bar", figsize=(10, 4), title="Average Benchmark Score by Version")

## Helper Functions for Metrics

In [ ]:
def rank_values(values):
    sorted_indices = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    position = 0
    while position < len(sorted_indices):
        next_position = position
        while next_position + 1 < len(sorted_indices) and values[sorted_indices[next_position + 1]] == values[sorted_indices[position]]:
            next_position += 1
        average_rank = (position + next_position + 2) / 2
        for current_position in range(position, next_position + 1):
            ranks[sorted_indices[current_position]] = average_rank
        position = next_position + 1
    return ranks

def pearson_correlation(xs, ys):
    if len(xs) < 2 or len(xs) != len(ys):
        return None
    mean_x = sum(xs) / len(xs)
    mean_y = sum(ys) / len(ys)
    numerator = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys))
    denominator_x = sum((x - mean_x) ** 2 for x in xs) ** 0.5
    denominator_y = sum((y - mean_y) ** 2 for y in ys) ** 0.5
    if denominator_x == 0 or denominator_y == 0:
        return None
    return numerator / (denominator_x * denominator_y)

def spearman_correlation(xs, ys):
    return pearson_correlation(rank_values(xs), rank_values(ys))


## Quantitative Metrics

In [ ]:
metrics_rows = []
for version in ["v3", "v4", "v5", "v6"]:
    version_df = benchmark_df[benchmark_df["version_label"] == version].copy()
    expected = version_df["expected_rating"].astype(float).tolist()
    predicted = version_df["rating"].astype(float).tolist()
    absolute_errors = [abs(p - e) for e, p in zip(expected, predicted)]
    exact_accuracy = sum(1 for e, p in zip(expected, predicted) if e == p) / len(expected)
    within_one = sum(1 for e, p in zip(expected, predicted) if abs(e - p) <= 1) / len(expected)
    over_scored = sum(1 for e, p in zip(expected, predicted) if p > e)
    under_scored = sum(1 for e, p in zip(expected, predicted) if p < e)
    metrics_rows.append({
        "version": version,
        "mae": sum(absolute_errors) / len(absolute_errors),
        "exact_accuracy": exact_accuracy,
        "within_one_accuracy": within_one,
        "over_scored": over_scored,
        "under_scored": under_scored,
        "pearson": pearson_correlation(expected, predicted),
        "spearman": spearman_correlation(expected, predicted),
    })

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.round(3)

In [ ]:
metrics_df.set_index("version")[["mae", "exact_accuracy", "within_one_accuracy", "pearson", "spearman"]].plot(kind="bar", figsize=(12, 5), title="Benchmark Metrics by Version")

## Case-by-Case Comparison

In [ ]:
case_pivot = benchmark_df.pivot_table(
    index=["case_id", "notes", "question_number", "question", "user_answer", "expected_rating"],
    columns="version_label",
    values="rating",
    aggfunc="first"
).reset_index()
case_pivot.head(10)

## Largest Disagreements Between Versions

In [ ]:
rating_columns = [column for column in ["v3", "v4", "v5", "v6"] if column in case_pivot.columns]
case_pivot["rating_spread"] = case_pivot[rating_columns].max(axis=1) - case_pivot[rating_columns].min(axis=1)
case_pivot.sort_values("rating_spread", ascending=False)[["case_id", "notes", "question", "user_answer", "expected_rating", "rating_spread"] + rating_columns].head(10)